## FPL Analysis

Scores and ranks all Premier League players for the upcoming gameweek.

Run all cells top to bottom. Sections:
1. **Config** — constants and scoring weights
2. **Data fetch** — live bootstrap API call
3. **Season weights** — blend function for ppg/form
4. **Load data** — read Excel snapshots
5. **Computations** — cleaning, FDR, normalization
6. **Scoring** — weighted score per position
7. **Recommendations** — top 15 per position
8. **Differentials** — low-ownership picks
9. **Watchlist & Context** — squad, watchlist, top N per position
10. **Squad Optimizer** — LP-based optimal 15-player squad
11. **Export** — save recommendations to markdown (optional)

In [37]:
import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import HTML

plt.style.use('dark_background')

### Config

Scoring weights, column lists, and constants. Edit weights here and re-run cells 6–9 to see updated rankings.

> Position weights sum to **0.80**. The remaining 0.20 comes from the season-aware consistency blend (ppg_last + ppg_current + form), merged at scoring time.

In [38]:
# Output settings
SAVE_OUTPUT = False  # Set to True to export recommendations to markdown

# Constants / Configs
POSITION_MASTER = {1: "GKP", 2: "DEF", 3: "MID", 4: "FWD"}
PLAYERS_NUMERIC_COLUMNS = [
    "form",
    "points_per_game",
    "ep_next",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "value_form",
    "value_season",
    "selected_by_percent",
    "expected_goals",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals_conceded",
    "clean_sheets_per_90",
    "saves_per_90",
    "ppg_last",
    "defensive_contribution",
]
PLAYERS_NORMALIZATION_COLUMNS = [
    "form",
    "points_per_game",
    "ep_next",
    "fdr_avg",
    "ict_index",
    "influence",
    "expected_goal_involvements",
    "expected_goals",
    "chance_of_playing_next_round",
    "clean_sheets_per_90",
    "saves_per_90",
    "threat",
    "ppg_last",
    "defensive_contribution",
]

# Position-specific scoring weights.
# NOTE: Each dict intentionally sums to 0.80, not 1.0.
# The remaining 0.20 is reserved for the season-aware consistency signal
# (ppg_last + ppg_current + form), which is blended separately in FORM_AND_PPG_WEIGHTS
# and merged in at scoring time. Together they sum to 1.0.
#
# Weights are backed by per-position correlation analysis (explore.py).
# Run `python explore.py --save` to regenerate heatmaps and retune if needed.
#
# Columns dampened by minutes_confidence (current season) before normalization:
# clean_sheets_per_90, saves_per_90, defensive_contribution, points_per_game, form
# ppg_last is dampened separately by ppg_last_confidence using last season's minutes (GW0).

GKP_WEIGHTS = {
    "ep_next": 0.25,
    "fdr_avg": 0.05,
    "chance_of_playing_next_round": 0.025,
    "clean_sheets_per_90": 0.35,  # strong signal (0.91 corr) — dampened by minutes_confidence before normalization
    "saves_per_90": 0.125,  # inverted + dampened — GKs on weak teams face more shots but concede more
}

DEF_WEIGHTS = {
    "ep_next": 0.35,  # bumped 0.25→0.35; validated by backtest (avg Spearman +0.004)
    "fdr_avg": 0.05,  # cut 0.15→0.05; corr 0.03 — near noise
    "chance_of_playing_next_round": 0.025,
    "ict_index": 0.075,  # split with influence; both 0.73 correlated
    "influence": 0.075,  # added; corr 0.60 — stronger than ict_index alone
    "defensive_contribution": 0.10,  # dampened by minutes_confidence before normalization
    "clean_sheets_per_90": 0.125,  # bumped 0.10→0.125
}

MID_WEIGHTS = {
    "ep_next": 0.275,  # bumped 0.07→0.275; validated by backtest (avg Spearman 0.606)
    "fdr_avg": 0.05,  # cut 0.18→0.05; corr -0.10 — negative for MIDs
    "chance_of_playing_next_round": 0.025,
    "ict_index": 0.11,  # split with influence; both 0.88 correlated
    "influence": 0.11,  # added; corr 0.91 — strongest MID signal
    "defensive_contribution": 0.10,  # dampened by minutes_confidence
    "expected_goal_involvements": 0.13,  # added; corr 0.67 — independent of influence
}


FWD_WEIGHTS = {
    "ep_next": 0.25,  # bumped 0.225→0.25
    "fdr_avg": 0.05,  # cut 0.10→0.05
    "chance_of_playing_next_round": 0.025,
    "threat": 0.15,  # cut 0.375→0.15; 0.88 correlated with ict_index
    "ict_index": 0.25,  # added; corr 0.81 — strongest independent FWD signal
    "expected_goals": 0.075,  # added; corr 0.52 — partial independent signal
}
PRESEASON_DATA_SHEET = "GW0"

# Number of upcoming GWs to average FDR over for recommendations.
# 1 = next GW only. Increase before blank/double GW weeks for better fixture awareness.
GAMEWEEK_WINDOW = 1  # GWs to average FDR over; used for fdr_avg scoring column and optimizer

### Fetch Live Data

Calls the FPL bootstrap-static API to detect the current gameweek.

In [39]:
# Get Next Gameweek
bootstrap_response = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/')
bootstrap_response.raise_for_status()
bootstrap_data = bootstrap_response.json()
events = bootstrap_data["events"]

next_gw_title: str | None = None
next_gw_id: int = -1
curr_gw_id: int = 0

for event in events:
  if event["is_next"]:
    next_gw_id = event["id"]
    curr_gw_id = next_gw_id - 1
    next_gw_title = event["name"]
    break

if next_gw_id == -1:
  raise RuntimeError('No upcoming gameweek found.')

# The upcoming GW we are predicting for
# Sheet was fetched before this GW was played
GAMEWEEK = f'GW{curr_gw_id}'

### Season-Aware Weight Blend

Shifts trust from last-season data toward current-season form as the season progresses.

| GWs played | ppg_last | ppg_current | form |
|---|---|---|---|
| 0 (pre-season) | 100% | 0% | 0% |
| 1–3 | 70% | 20% | 10% |
| 4–6 | 40% | 30% | 30% |
| 7–10 | 10% | 40% | 50% |
| 11+ | 0% | 45% | 55% |

In [40]:
def get_season_weights(gws_played: int) -> tuple[float, float, float]:
  """Returns a tuple indicating the weight distribution of last season's points per game, and current season's points per game and 
  and form.
    
    (last_season_ppg, current_season_ppg, current_season_form)
    """
  if gws_played < 1:
    return 1, 0, 0
  elif gws_played < 4:
    return 0.7, 0.2, 0.1
  elif gws_played < 7:
    return 0.4, 0.3, 0.3
  elif gws_played < 11:
    return 0.1, 0.4, 0.5
  else:
    return 0, 0.45, 0.55

### Load Data

Reads the latest GW sheet from all three Excel files.

In [41]:
# Load Data.

# Guard: verify GAMEWEEK sheet exists before loading.
# If fetch.py hasn't been run yet (or failed), the sheet won't exist and read_excel
# would throw an opaque KeyError. This surfaces a clear, actionable error instead.
xl_players = pd.ExcelFile('./data/players_master.xlsx')
xl_teams = pd.ExcelFile('./data/teams_master.xlsx')
if GAMEWEEK not in xl_players.sheet_names:
    raise RuntimeError(f"Sheet '{GAMEWEEK}' not found in players_master.xlsx. Run fetch.py first. Available: {xl_players.sheet_names}")
if GAMEWEEK not in xl_teams.sheet_names:
    raise RuntimeError(f"Sheet '{GAMEWEEK}' not found in teams_master.xlsx. Run fetch.py first. Available: {xl_teams.sheet_names}")

players_df = pd.read_excel(xl_players, sheet_name=GAMEWEEK)
players_df_last_season = pd.read_excel(xl_players, sheet_name=PRESEASON_DATA_SHEET)
teams_df = pd.read_excel(xl_teams, sheet_name=GAMEWEEK)
fixtures_df = pd.read_excel('./data/fixtures_master.xlsx', sheet_name='Fixtures')
current_gw_fixtures_df = fixtures_df[fixtures_df['event'] == next_gw_id]

### Computations

Cleaning, type casting, FDR calculation, `minutes_confidence` dampening, and min-max normalization.

In [42]:
# Computations

# Join last season's ppg from GW0 (pre-season snapshot) using 'code' as the stable cross-season ID.
# 'id' can change between seasons; 'code' is permanent.
# New players with no GW0 entry (e.g. promoted clubs, transfers from abroad) get ppg_last = 0.
players_df_last_season = players_df_last_season[['points_per_game', 'minutes', 'code']].rename(columns={'points_per_game': 'ppg_last', 'minutes': 'minutes_last'})
players_df = players_df.merge(players_df_last_season, on='code', how='left')
players_df['ppg_last'] = players_df['ppg_last'].fillna(0)
players_df['minutes_last'] = players_df['minutes_last'].fillna(0)

players_df['full_name'] = players_df['first_name'] + ' ' + players_df['second_name']
players_df['position'] = players_df['element_type'].map(POSITION_MASTER)
players_df['team_name'] = players_df['team'].map(dict(zip(teams_df['id'], teams_df['name'])))

# FPL returns some numeric columns as strings (e.g. form, ep_next, ict_index).
# Cast them to float before any numeric operation; coerce invalid values to NaN.
for col in PLAYERS_NUMERIC_COLUMNS:
  players_df[col] = pd.to_numeric(players_df[col], errors='coerce')

# chance_of_playing_next_round is null (not 0) when a player has no injury concern.
# Treat null as 100% available.
players_df['chance_of_playing_next_round'] = players_df['chance_of_playing_next_round'].fillna(100)
# ep_next is null pre-season (no GW played yet). Treat as 0 expected points.
players_df['ep_next'] = players_df['ep_next'].fillna(0)

# Keep only players with status a (available), i (injured), d (doubtful).
# Drop s (suspended) and u (unavailable/left club) — not selectable.
players_df = players_df[players_df['status'].isin(['a', 'i', 'd'])]

# minutes_confidence: dampens per-90 and contribution stats for players with few appearances.
# Formula: minutes_played / (gws_played * 90), capped at 1.0.
# Pre-season (curr_gw_id = 0): use full season (38 * 90) as denominator to avoid division by zero.
# A player who played every minute gets 1.0 (full trust); 1-game player gets ~0.026 pre-season.
conf_denominator = curr_gw_id * 90 if curr_gw_id > 0 else 38 * 90
players_df['minutes_confidence'] = (players_df['minutes'] / conf_denominator).clip(upper=1.0)

# Apply confidence multiplier before normalization — dampens small-sample outliers.
# All are per-game averages that produce inflated values for players with very few appearances.
#
# ppg_last uses minutes_last (from GW0) with a full-season denominator (38 * 90)
# since it reflects last season's data — current season's minutes are irrelevant for it.
ppg_last_confidence = (players_df['minutes_last'] / (38 * 90)).clip(upper=1.0)
players_df['ppg_last'] = players_df['ppg_last'] * ppg_last_confidence

# All other columns use current-season minutes_confidence.
for col in ['clean_sheets_per_90', 'saves_per_90', 'defensive_contribution', 'points_per_game', 'form']:
  players_df[col] = players_df[col] * players_df['minutes_confidence']

# FPL stores cost * 10 (e.g. 60 = £6.0m). Derive actual cost and value metric.
players_df['cost'] = players_df['now_cost'] / 10
players_df['points_per_euro'] = players_df['total_points'] / players_df['cost']

# Build FDR for the immediate next GW — display context only.
home = current_gw_fixtures_df[['team_h', 'team_h_difficulty']].rename(columns={'team_h': 'team_id', 'team_h_difficulty': 'fdr'})
away = current_gw_fixtures_df[['team_a', 'team_a_difficulty']].rename(columns={'team_a': 'team_id', 'team_a_difficulty': 'fdr'})
fdr_df = pd.concat([home, away])
players_df['fdr'] = players_df['team'].map(dict(zip(fdr_df['team_id'], fdr_df['fdr'])))

# fdr_avg: average FDR over GAMEWEEK_WINDOW upcoming GWs — used in scoring and optimizer.
# groupby mean handles double GW weeks where a team has multiple fixtures in the window.
fdr_avg_window = min(GAMEWEEK_WINDOW, 38 - curr_gw_id)
fdr_avg_fixtures = fixtures_df[fixtures_df['event'].between(next_gw_id, next_gw_id + fdr_avg_window - 1)]
home_avg = fdr_avg_fixtures[['team_h', 'team_h_difficulty']].rename(columns={'team_h': 'team_id', 'team_h_difficulty': 'fdr'})
away_avg = fdr_avg_fixtures[['team_a', 'team_a_difficulty']].rename(columns={'team_a': 'team_id', 'team_a_difficulty': 'fdr'})
fdr_avg_map = pd.concat([home_avg, away_avg]).groupby('team_id')['fdr'].mean().to_dict()
players_df['fdr_avg'] = players_df['team'].map(fdr_avg_map)

# Rebuild as a contiguous copy to resolve pandas memory fragmentation warning.
players_df = players_df.copy()

# Min-max normalize all scoring columns to 0-1 so weights are meaningful across different scales.
# FDR and saves_per_90 are inverted: lower raw value = better for the player.
# Guard: if all values are identical (e.g. form = 0.0 pre-season), denominator = 1 → everyone scores 0.
for col in PLAYERS_NORMALIZATION_COLUMNS:
  col_min = players_df[col].min()
  col_max = players_df[col].max()
  denominator = col_max - col_min if col_max != col_min else 1
  if col in ['fdr_avg', 'saves_per_90']:
    players_df[f'{col}_norm'] = 1 - ((players_df[col] - col_min) / denominator)
  else:
    players_df[f'{col}_norm'] = (players_df[col] - col_min) / denominator


/var/folders/xq/b2slm4tn4zz2nv3r96657h0m0000gn/T/ipykernel_27400/3980540874.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  players_df['fdr_avg'] = players_df['team'].map(fdr_avg_map)


### Scoring

Applies position-specific weights to normalized columns. Each position uses different factors backed by correlation analysis (`explore.py`).

In [43]:
# Scoring

# Resolve season-aware blend weights for the consistency signal.
# Returns (ppg_last_w, ppg_curr_w, form_curr_w) based on how many GWs have been played.
# All three sum to 1.0, and the combined FORM_AND_PPG budget is 0.20 of the total score.
[ppg_last_w, ppg_curr_w, form_curr_w] = get_season_weights(curr_gw_id)

FORM_AND_PPG_WEIGHTS = {
  'ppg_last': 0.20 * ppg_last_w,        # last season's ppg from GW0 snapshot
  'points_per_game': 0.20 * ppg_curr_w, # current season running average
  'form': 0.20 * form_curr_w             # rolling recent GW average
}

# Merge position-specific weights with the shared consistency signal.
# Each merged dict sums to 1.0: position weights (0.80) + FORM_AND_PPG (0.20).
gpk_weights = {**GKP_WEIGHTS, **FORM_AND_PPG_WEIGHTS}
def_weights = {**DEF_WEIGHTS, **FORM_AND_PPG_WEIGHTS}
mid_weights = {**MID_WEIGHTS, **FORM_AND_PPG_WEIGHTS}
fwd_weights = {**FWD_WEIGHTS, **FORM_AND_PPG_WEIGHTS}

# Compute next_gw_score per position using a weighted sum of normalized columns.
mask = players_df['position'] == 'GKP'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in gpk_weights.items()])
mask = players_df['position'] == 'DEF'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in def_weights.items()])
mask = players_df['position'] == 'MID'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in mid_weights.items()])
mask = players_df['position'] == 'FWD'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in fwd_weights.items()])


### Recommendations

Top 15 players per position ranked by `next_gw_score`.

In [44]:
# Visualizations
 
rec_cols = ['full_name', 'team_name', 'position', 'fdr', 'fdr_avg', 'minutes', 'cost', 'total_points', 'points_per_euro', 'price_change_percent', 'next_gw_score']
display_format = { 
  'next_gw_score': '{:.2f}',
   'cost': '£{:.1f}',
   'position_rank': '{:.0f}',
   'clean_sheets_per_90': '{:.2f}',
   'saves_per_90': '{:.2f}',
   'points_per_euro': '{:.2f}',
   'points_per_game': '{:.2f}',
   'differential_score': '{:.2f}',
   'selected_by_percent': '{:.1f}%',
   'ep_next': '{:.2f}',
   'fdr': '{:.0f}',
  'fdr_avg': '{:.2f}',
  'price_change_percent': '{:+.1f}%',
}
recommendations = players_df.sort_values(by=['position', 'next_gw_score'], ascending=False).groupby('position').head(15)[rec_cols]

for pos in POSITION_MASTER.values():
  temp_df = recommendations[recommendations['position'] == pos]
  display(
    temp_df.sort_values(by='next_gw_score', ascending=False)
    .style
    .hide(axis='index')
    .set_caption(pos)
    .background_gradient(subset=['next_gw_score'], cmap='Greens')
    .format({**display_format})
  )

full_name,team_name,position,fdr,fdr_avg,minutes,cost,total_points,points_per_euro,price_change_percent,next_gw_score
David Raya Martín,Arsenal,GKP,4,4.00,180,£6.0,12,2.00,+8.6%,0.75
Konstantinos Tzolakis,Hull City,GKP,3,3.00,180,£4.6,20,4.35,+1.9%,0.67
Robin Roefs,Sunderland,GKP,3,3.00,180,£5.0,7,1.40,-39.8%,0.51
Bart Verbruggen,Brighton,GKP,2,2.00,180,£4.5,6,1.33,+33.0%,0.51
Caoimhín Kelleher,Brentford,GKP,2,2.00,180,£5.0,9,1.80,+33.6%,0.51
Jordan Pickford,Everton,GKP,4,4.00,180,£5.5,10,1.82,+9.7%,0.44
James Trafford,Leeds,GKP,3,3.00,180,£5.0,12,2.40,+24.4%,0.42
Lukás Hornícek,Newcastle,GKP,3,3.00,180,£5.0,8,1.60,+10.3%,0.33
Gianluigi Donnarumma,Man City,GKP,2,2.00,180,£5.5,3,0.55,-24.3%,0.27
Nick Pope,Newcastle,GKP,3,3.00,0,£5.0,0,0.00,-81.4%,0.24


full_name,team_name,position,fdr,fdr_avg,minutes,cost,total_points,points_per_euro,price_change_percent,next_gw_score
James Tarkowski,Everton,DEF,4,4.00,180,£6.0,18,3.00,+51.4%,0.62
John Egan,Hull City,DEF,3,3.00,180,£4.0,17,4.25,+66.7%,0.61
Riccardo Calafiori,Arsenal,DEF,4,4.00,170,£5.6,20,3.57,+100.8%,0.60
Semi Ajayi,Hull City,DEF,3,3.00,153,£4.1,20,4.88,+22.4%,0.59
Gabriel dos Santos Magalhães,Arsenal,DEF,4,4.00,180,£8.0,13,1.62,-31.8%,0.56
Benjamin White,Arsenal,DEF,4,4.00,180,£5.5,18,3.27,+62.3%,0.55
Michael Kayode,Brentford,DEF,2,2.00,162,£4.6,15,3.26,+32.2%,0.52
Maxim De Cuyper,Brighton,DEF,2,2.00,167,£4.7,17,3.62,+25.1%,0.52
Lewis Hall,Newcastle,DEF,3,3.00,180,£5.1,14,2.75,+9.7%,0.51
Kristoffer Ajer,Brentford,DEF,2,2.00,180,£4.5,11,2.44,+42.6%,0.48


full_name,team_name,position,fdr,fdr_avg,minutes,cost,total_points,points_per_euro,price_change_percent,next_gw_score
Bruno Borges Fernandes,Man Utd,MID,3,3.00,180,£12.0,25,2.08,-12.8%,0.92
Rayan Cherki,Man City,MID,2,2.00,108,£7.7,22,2.86,+73.7%,0.63
Bukayo Saka,Arsenal,MID,4,4.00,157,£9.5,20,2.11,+18.5%,0.61
Cole Palmer,Chelsea,MID,5,5.00,172,£9.6,20,2.08,+9.5%,0.61
Bryan Mbeumo,Man Utd,MID,3,3.00,180,£8.0,13,1.62,-90.2%,0.57
Dominik Szoboszlai,Liverpool,MID,2,2.00,180,£7.0,12,1.71,+37.6%,0.57
Morgan Rogers,Chelsea,MID,5,5.00,171,£7.5,13,1.73,+67.0%,0.56
Cody Gakpo,Liverpool,MID,2,2.00,160,£7.0,17,2.43,+100.2%,0.56
Anton Stach,Leeds,MID,3,3.00,180,£6.0,17,2.83,+31.2%,0.55
Keane Lewis-Potter,Brentford,MID,2,2.00,177,£5.5,16,2.91,+42.0%,0.55


full_name,team_name,position,fdr,fdr_avg,minutes,cost,total_points,points_per_euro,price_change_percent,next_gw_score
Erling Haaland,Man City,FWD,2,2.00,180,£15.5,15,0.97,+52.8%,0.70
João Pedro Junqueira de Jesus,Chelsea,FWD,5,5.00,180,£7.7,20,2.60,+5.7%,0.70
Alexander Isak,Liverpool,FWD,2,2.00,180,£9.0,10,1.11,-6.3%,0.56
Thierno Barry,Everton,FWD,4,4.00,147,£5.5,10,1.82,+83.5%,0.51
Yoane Wissa,Newcastle,FWD,3,3.00,171,£6.1,12,1.97,+65.8%,0.46
Igor Thiago Nascimento Rodrigues,Brentford,FWD,2,2.00,172,£8.0,2,0.25,-61.4%,0.40
Dominic Calvert-Lewin,Leeds,FWD,3,3.00,180,£6.0,9,1.50,-16.0%,0.40
Gonzalo García,Fulham,FWD,3,3.00,180,£6.0,8,1.33,+86.2%,0.36
Francisco Evanilson de Lima Barbosa,Bournemouth,FWD,3,3.00,154,£6.0,9,1.50,+39.3%,0.34
Igor Jesus Maciel da Cruz,Nott'm Forest,FWD,3,3.00,180,£5.9,3,0.51,-5.0%,0.34


### Differential Picks

Low-ownership players with strong scoring potential. `differential_score = next_gw_score × (1 - ownership_norm)`, where ownership is normalized per position.

In [45]:
# Differential Player (By Position)

col = 'selected_by_percent'
diff_cols = rec_cols + ['selected_by_percent', 'differential_score']
for pos in POSITION_MASTER.values():
  temp_df = players_df[players_df['position'] == pos]
  mask = players_df['position'] == pos
  if temp_df.empty:
    continue
  col_min = temp_df[col].min()
  col_max = temp_df[col].max()
  denominator = col_max - col_min if col_max != col_min else 1
  players_df.loc[mask, 'selected_by_percent_norm'] = 1 - ((temp_df[col] - col_min) / denominator)
  
players_df['differential_score'] = players_df['next_gw_score'] * players_df['selected_by_percent_norm']

differentials = players_df.sort_values(by=['position', 'differential_score'], ascending=False).groupby('position').head(15)[diff_cols]

for pos in POSITION_MASTER.values():
  temp_df = differentials[differentials['position'] == pos]
  if temp_df.empty:
    continue
  display(
    temp_df[diff_cols]
    .sort_values(by='differential_score', ascending=False)
    .style
    .hide(axis='index')
    .set_caption(f"{pos} ({len(players_df[players_df['position'] == pos])})")
    .background_gradient(subset=['differential_score'], cmap='Greens')
    .format({**display_format})
  )


full_name,team_name,position,fdr,fdr_avg,minutes,cost,total_points,points_per_euro,price_change_percent,next_gw_score,selected_by_percent,differential_score
Konstantinos Tzolakis,Hull City,GKP,3,3.00,180,£4.6,20,4.35,+1.9%,0.67,7.3%,0.54
Robin Roefs,Sunderland,GKP,3,3.00,180,£5.0,7,1.40,-39.8%,0.51,3.5%,0.47
Caoimhín Kelleher,Brentford,GKP,2,2.00,180,£5.0,9,1.80,+33.6%,0.51,7.4%,0.41
James Trafford,Leeds,GKP,3,3.00,180,£5.0,12,2.40,+24.4%,0.42,5.1%,0.36
Jordan Pickford,Everton,GKP,4,4.00,180,£5.5,10,1.82,+9.7%,0.44,8.7%,0.34
Lukás Hornícek,Newcastle,GKP,3,3.00,180,£5.0,8,1.60,+10.3%,0.33,1.1%,0.32
Nick Pope,Newcastle,GKP,3,3.00,0,£5.0,0,0.00,-81.4%,0.24,1.1%,0.24
Karl Darlow,Man Utd,GKP,3,3.00,0,£4.5,0,0.00,-6.8%,0.23,0.0%,0.23
Michele Di Gregorio,Bournemouth,GKP,3,3.00,0,£4.5,0,0.00,+0.0%,0.22,0.0%,0.22
Bart Verbruggen,Brighton,GKP,2,2.00,180,£4.5,6,1.33,+33.0%,0.51,21.6%,0.22


full_name,team_name,position,fdr,fdr_avg,minutes,cost,total_points,points_per_euro,price_change_percent,next_gw_score,selected_by_percent,differential_score
John Egan,Hull City,DEF,3,3.00,180,£4.0,17,4.25,+66.7%,0.61,4.7%,0.54
Semi Ajayi,Hull City,DEF,3,3.00,153,£4.1,20,4.88,+22.4%,0.59,8.5%,0.47
James Tarkowski,Everton,DEF,4,4.00,180,£6.0,18,3.00,+51.4%,0.62,10.5%,0.47
Benjamin White,Arsenal,DEF,4,4.00,180,£5.5,18,3.27,+62.3%,0.55,7.3%,0.46
James Justin,Leeds,DEF,3,3.00,180,£4.5,12,2.67,+15.5%,0.46,1.4%,0.44
Kristoffer Ajer,Brentford,DEF,2,2.00,180,£4.5,11,2.44,+42.6%,0.48,5.0%,0.43
Nathan Collins,Brentford,DEF,2,2.00,180,£5.5,9,1.64,+11.0%,0.45,2.3%,0.43
Nobel Mendy,Hull City,DEF,3,3.00,121,£4.0,16,4.00,+50.6%,0.48,5.5%,0.42
Ryan Giles,Hull City,DEF,3,3.00,174,£4.0,12,3.00,+7.5%,0.43,1.1%,0.42
Lewis Hall,Newcastle,DEF,3,3.00,180,£5.1,14,2.75,+9.7%,0.51,8.9%,0.41


full_name,team_name,position,fdr,fdr_avg,minutes,cost,total_points,points_per_euro,price_change_percent,next_gw_score,selected_by_percent,differential_score
Keane Lewis-Potter,Brentford,MID,2,2.00,177,£5.5,16,2.91,+42.0%,0.55,1.9%,0.53
Anton Stach,Leeds,MID,3,3.00,180,£6.0,17,2.83,+31.2%,0.55,2.9%,0.52
Bukayo Saka,Arsenal,MID,4,4.00,157,£9.5,20,2.11,+18.5%,0.61,10.6%,0.48
Alex Scott,Bournemouth,MID,3,3.00,176,£6.0,14,2.33,+12.7%,0.50,1.9%,0.48
Cody Gakpo,Liverpool,MID,2,2.00,160,£7.0,17,2.43,+100.2%,0.56,8.2%,0.46
Kevin Schade,Brentford,MID,2,2.00,177,£6.0,13,2.17,+23.1%,0.50,4.1%,0.46
Kiernan Dewsbury-Hall,Everton,MID,4,4.00,180,£6.5,13,2.00,+38.6%,0.48,4.4%,0.44
Vitaly Janelt,Brentford,MID,2,2.00,180,£5.0,10,2.00,+19.3%,0.44,0.6%,0.44
Marcus Tavernier,Bournemouth,MID,3,3.00,180,£6.0,11,1.83,+28.7%,0.45,2.3%,0.43
Granit Xhaka,Sunderland,MID,3,3.00,180,£5.5,15,2.73,+13.4%,0.48,4.6%,0.43


full_name,team_name,position,fdr,fdr_avg,minutes,cost,total_points,points_per_euro,price_change_percent,next_gw_score,selected_by_percent,differential_score
Thierno Barry,Everton,FWD,4,4.00,147,£5.5,10,1.82,+83.5%,0.51,4.5%,0.48
Alexander Isak,Liverpool,FWD,2,2.00,180,£9.0,10,1.11,-6.3%,0.56,16.6%,0.43
Yoane Wissa,Newcastle,FWD,3,3.00,171,£6.1,12,1.97,+65.8%,0.46,12.5%,0.38
Gonzalo García,Fulham,FWD,3,3.00,180,£6.0,8,1.33,+86.2%,0.36,4.6%,0.33
Francisco Evanilson de Lima Barbosa,Bournemouth,FWD,3,3.00,154,£6.0,9,1.50,+39.3%,0.34,3.2%,0.33
Igor Jesus Maciel da Cruz,Nott'm Forest,FWD,3,3.00,180,£5.9,3,0.51,-5.0%,0.34,3.1%,0.32
Igor Thiago Nascimento Rodrigues,Brentford,FWD,2,2.00,172,£8.0,2,0.25,-61.4%,0.40,14.3%,0.32
Emersonn Correia da Silva,Ipswich Town,FWD,4,4.00,110,£5.5,10,1.82,+12.1%,0.33,1.6%,0.32
Oli McBurnie,Hull City,FWD,3,3.00,180,£5.5,6,1.09,+30.0%,0.30,2.1%,0.29
Wilson Isidor,Sunderland,FWD,3,3.00,49,£5.5,9,1.64,+2.2%,0.29,1.1%,0.28


### Watchlist & My Team in Context

Configurable view — shows top N players per position with your squad, watchlist, and team filter players highlighted. All lists are optional; set `show_top_n = False` to show only your listed players.

In [ ]:
# Watchlist / My Team in Context

# --- Config ---
show_top_n = True   # include top N players per position
top_n = 15          # how many top players to show per position

# Lists (all optional — empty list = not used)
# add player codes

my_team_players = [
    85633,   # Sels
    489639,  # Verbruggen
    477424,  # Gvardiol
    607464,  # Kayode
    198869,  # White
    466075,  # Calafiori
    146426,  # Ajayi
    141746,  # B.Fernandes
    513418,  # Schade
    494595,  # Wirtz
    432714,  # McAtee
    448047,  # Enzo Fernandez
    223094,  # Haaland
    177815,  # Calvert-Lewin
    586309,  # Barry
]

watchlist_players = []


# add team_code values
watchlist_teams   = []

# --- Highlight priority: my_team > watchlist > team ---
def highlight(row):
  if row["code"] in my_team_players:
    return ["background-color: #8b3a3a"] * len(row)
  if row["code"] in watchlist_players:
    return ["background-color: #3d1515"] * len(row)
  if row["team_code"] in watchlist_teams:
    return ["background-color: #0d2137"] * len(row)
  return [""] * len(row)

players_df["position_rank"] = players_df.groupby("position")["next_gw_score"].rank(ascending=False, method="min")

# --- Build combined column list ---
view_cols = ["code", "team_code"] + rec_cols + ["position_rank", "differential_score", "selected_by_percent"]

# --- Build legend (only show labels for non-empty sources) ---
legend_items = []
if len(my_team_players):   legend_items.append("<span style='background-color:#8b3a3a; padding:3px 10px; border-radius:3px; margin-right:8px;'>&#9632; My team</span>")
if len(watchlist_players): legend_items.append("<span style='background-color:#3d1515; padding:3px 10px; border-radius:3px; margin-right:8px;'>&#9632; Watchlist</span>")
if len(watchlist_teams):   legend_items.append("<span style='background-color:#0d2137; padding:3px 10px; border-radius:3px;'>&#9632; Team filter</span>")
if legend_items:
  display(HTML(f"<div style='margin-bottom:12px; font-size:13px;'>{''.join(legend_items)}</div>"))

# --- Build per-position dataframe ---
for pos in POSITION_MASTER.values():
  pos_df = players_df[players_df["position"] == pos]

  # Start with top N if enabled
  if show_top_n:
    base_df = pos_df.nlargest(top_n, "next_gw_score")
  else:
    base_df = pos_df.iloc[0:0]  # empty

  # Union in all explicitly listed players
  extra_df = pos_df[pos_df["code"].isin(my_team_players)
                    | pos_df["code"].isin(watchlist_players)
                    | pos_df["team_code"].isin(watchlist_teams)]

  temp_df = pd.concat([base_df, extra_df]).drop_duplicates(subset=["code"]).sort_values("position_rank")

  if temp_df.empty:
    continue

  n_extra = len(temp_df) - len(base_df)
  caption = f"{pos} — top {top_n}" if show_top_n else f"{pos}"
  if n_extra > 0: 
    caption += f" + {n_extra} extra"

  display(
    temp_df[view_cols]
    .style
    .apply(highlight, axis=1)
    .hide(axis="index")
    .hide(subset=["code", "team_code"], axis="columns")
    .set_caption(caption)
    .background_gradient(subset=["next_gw_score"], cmap="Greens")
    .format({**display_format})
  )

### Squad Optimizer

Uses linear programming (PuLP) to find the optimal 15-player squad under FPL constraints: 2 GKP, 5 DEF, 5 MID, 3 FWD, max 3 per team, total cost ≤ £100.0m.

In [47]:
# Squad Optimizer — best 15-player squad under FPL constraints.
from pulp import PULP_CBC_CMD, LpBinary, LpMaximize, LpProblem, LpVariable, lpSum

# Config.
BUDGET = 100.0  # max squad cost in £m
POSITION_COUNTS = {"GKP": 2, "DEF": 5, "MID": 5, "FWD": 3}
MAX_PER_TEAM = 3
# GAMEWEEK_WINDOW defined in config cell.
FDR_WEIGHT = 0.3     # weight for fdr_avg_norm in objective 2
QUALITY_WEIGHT = 0.7  # weight for next_gw_score in objective 2; FDR_WEIGHT (0.3) covers the rest

MUST_HAVE_PLAYERS = []
AVOID_PLAYERS = []

def highlight_forced(row):
  if row["code"] in MUST_HAVE_PLAYERS:
    return ["background-color: #8b3a3a"] * len(row)
  return [""] * len(row)

# Build lookup dicts from players_df.
# TODO: Loosen this filter once more GWs are played (early season: rotation players
# who blanked one GW get excluded unfairly). Consider switching to minutes > 0
# or status == 'a' as the season progresses.
# players_df_copy = players_df[
#   (players_df['total_points'] > 2) &
#   (players_df['minutes'] > 80)
# ]
players_df_copy = players_df[players_df['minutes'] > 0]

player_codes = players_df_copy["code"].tolist()
ep_next = dict(zip(players_df_copy["code"], players_df_copy["ep_next"]))
player_cost = dict(zip(players_df_copy["code"], players_df_copy["cost"]))
fdr_avg = dict(zip(players_df_copy["code"], players_df_copy["fdr_avg"]))
next_gw_score_dict = dict(zip(players_df_copy["code"], players_df_copy["next_gw_score"]))
fdr_avg_norm_dict = dict(zip(players_df_copy["code"], players_df_copy["fdr_avg_norm"]))

position_to_codes = players_df_copy.groupby("position")["code"].apply(list).to_dict()
team_to_codes = players_df_copy.groupby("team_code")["code"].apply(list).to_dict()

# Create LP problem.
prob = LpProblem("fpl_squad", LpMaximize)

# Decision variables: x[code] = 1 if player selected, 0 otherwise.
x = LpVariable.dicts("select", player_codes, cat=LpBinary)

# Objective 1: maximize total ep_next.
# prob += lpSum([x[code] * ep_next[code] for code in player_codes])


# Objective 2: next_gw_score + fdr_avg_norm — best squad over GAMEWEEK_WINDOW GWs.
# next_gw_score captures player quality (form, ict, ep_next, availability).
# fdr_avg_norm is already inverted (higher = easier fixtures) and averaged over the window.
# Changing GAMEWEEK_WINDOW now affects which players get selected.
prob += lpSum([
    x[code] * (QUALITY_WEIGHT * next_gw_score_dict[code] + FDR_WEIGHT * fdr_avg_norm_dict[code])
    for code in player_codes
])

# Constraints: position counts.
for pos, count in POSITION_COUNTS.items():
  prob += lpSum([x[code] for code in position_to_codes[pos]]) == count

# Constraints: max 3 per team.
for codes in team_to_codes.values():
  prob += lpSum([x[code] for code in codes]) <= MAX_PER_TEAM

# Constraints: budget.
prob += lpSum([x[code] * player_cost[code] for code in player_codes]) <= BUDGET

# Force Inclusion/Exclusion of players.
for code in MUST_HAVE_PLAYERS:
  prob += x[code] == 1
for code in AVOID_PLAYERS:
  prob += x[code] == 0

# Solve.
prob.solve(PULP_CBC_CMD(msg=False))

# Extract selected players.
selected_codes = [code for code in player_codes if x[code].value() == 1]
squad_df = players_df_copy[players_df_copy["code"].isin(selected_codes)].copy()

# Display result.
total_cost = squad_df["cost"].sum()
total_ep = squad_df["ep_next"].sum()
total_ppg = squad_df["points_per_game"].sum()
avg_fdr = squad_df["fdr_avg"].mean()

print(f"Optimal Squad: {len(selected_codes)} players | Cost: £{total_cost:.1f}m | EP: {total_ep:.2f} | PPG: {total_ppg:.2f} | Avg FDR: {avg_fdr:.2f}")
print()

squad_cols = ["full_name", "code", "team_name", "position", "fdr", "fdr_avg", "minutes", "cost", "total_points", "points_per_euro", "ep_next",
"selected_by_percent", "points_per_game", "next_gw_score"]
display(
  squad_df[squad_cols]
  .sort_values(["position", "next_gw_score"], key=lambda c: c.map({"GKP": 0, "DEF": 1, "MID": 2, "FWD": 3}) if c.name == "position" else c,
ascending=[True, False])
  .style
  .apply(highlight_forced, axis=1)
  .hide(axis="index")
  .background_gradient(subset=["next_gw_score"], cmap="Greens")
  .format({**display_format})
)

Optimal Squad: 15 players | Cost: £99.8m | EP: 115.50 | PPG: 107.80 | Avg FDR: 2.33



full_name,code,team_name,position,fdr,fdr_avg,minutes,cost,total_points,points_per_euro,ep_next,selected_by_percent,points_per_game,next_gw_score
Konstantinos Tzolakis,473284,Hull City,GKP,3,3.00,180,£4.6,20,4.35,10.00,7.3%,10.00,0.67
Bart Verbruggen,489639,Brighton,GKP,2,2.00,180,£4.5,6,1.33,3.00,21.6%,3.00,0.51
John Egan,108416,Hull City,DEF,3,3.00,180,£4.0,17,4.25,8.50,4.7%,8.50,0.61
Semi Ajayi,146426,Hull City,DEF,3,3.00,153,£4.1,20,4.88,10.00,8.5%,8.50,0.59
Michael Kayode,607464,Brentford,DEF,2,2.00,162,£4.6,15,3.26,7.50,9.4%,6.75,0.52
Maxim De Cuyper,465730,Brighton,DEF,2,2.00,167,£4.7,17,3.62,8.50,15.5%,7.89,0.52
Kristoffer Ajer,191866,Brentford,DEF,2,2.00,180,£4.5,11,2.44,5.50,5.0%,5.50,0.48
Bruno Borges Fernandes,141746,Man Utd,MID,3,3.00,180,£12.0,25,2.08,12.50,48.6%,12.50,0.92
Rayan Cherki,466052,Man City,MID,2,2.00,108,£7.7,22,2.86,11.00,27.0%,6.60,0.63
Dominik Szoboszlai,424876,Liverpool,MID,2,2.00,180,£7.0,12,1.71,6.00,41.4%,6.00,0.57


### Export to Markdown

Saves recommendations, differentials, and optimal squad to a markdown file when `SAVE_OUTPUT = True`.

In [48]:
# Export recommendations to markdown file.
# Only runs when SAVE_OUTPUT = True in the config cell.

if SAVE_OUTPUT:
    from datetime import datetime, timedelta, timezone
    from pathlib import Path

    # Setup
    today = datetime.now(tz=timezone(offset=timedelta(hours=5, minutes=30))).strftime('%d-%m-%Y')
    reports_dir = Path('data/reports')
    reports_dir.mkdir(parents=True, exist_ok=True)
    filename = reports_dir / f'gw{next_gw_id}-{today}.md'

    # Column subsets for export (cleaner than full DataFrame)
    rec_export_cols = ['full_name', 'team_name', 'position', 'fdr', 'fdr_avg', 'cost', 'total_points', 'points_per_euro', 'price_change_percent', 'next_gw_score']
    diff_export_cols = ['full_name', 'team_name', 'position', 'fdr', 'cost', 'total_points', 'next_gw_score', 'selected_by_percent', 'differential_score']
    squad_export_cols = ['full_name', 'team_name', 'position', 'fdr', 'fdr_avg', 'cost', 'ep_next', 'points_per_game', 'next_gw_score']

    # Build markdown content
    lines = [
        f'# FPL Analysis — Gameweek {next_gw_id}',
        '',
        f'Generated: {today}',
        '',
        f'Data sheet: `{GAMEWEEK}` (stats after GW{curr_gw_id})',
        '',
        f'---',  # noqa: F541
        '',
        f'## Recommendations',  # noqa: F541
        '',
        f'Top 15 players per position ranked by `next_gw_score`.',  # noqa: F541
        '',
    ]

    # Recommendations per position
    for pos in POSITION_MASTER.values():
        pos_df = players_df[players_df['position'] == pos].nlargest(15, 'next_gw_score')[rec_export_cols].copy()
        pos_df['cost'] = pos_df['cost'].apply(lambda x: f'£{x:.1f}')
        pos_df['points_per_euro'] = pos_df['points_per_euro'].apply(lambda x: f'{x:.2f}')
        pos_df['next_gw_score'] = pos_df['next_gw_score'].apply(lambda x: f'{x:.2f}')
        lines.append(f'### {pos}')
        lines.append('')
        lines.append(pos_df.to_markdown(index=False))
        lines.append('')

    # Differentials
    lines.append('---')
    lines.append('')
    lines.append('## Differentials')
    lines.append('')
    lines.append('Low-ownership players with strong scoring potential. `differential_score = next_gw_score × (1 - ownership_norm)`.')
    lines.append('')

    for pos in POSITION_MASTER.values():
        pos_df = players_df[players_df['position'] == pos].nlargest(15, 'differential_score')[diff_export_cols].copy()
        pos_df['cost'] = pos_df['cost'].apply(lambda x: f'£{x:.1f}')
        pos_df['next_gw_score'] = pos_df['next_gw_score'].apply(lambda x: f'{x:.2f}')
        pos_df['selected_by_percent'] = pos_df['selected_by_percent'].apply(lambda x: f'{x:.1f}%')
        pos_df['differential_score'] = pos_df['differential_score'].apply(lambda x: f'{x:.2f}')
        lines.append(f'### {pos}')
        lines.append('')
        lines.append(pos_df.to_markdown(index=False))
        lines.append('')

    # Optimal Squad
    lines.append('---')
    lines.append('')
    lines.append('## Optimal Squad')
    lines.append('')
    lines.append(f'Best 15-player squad under FPL constraints (2 GKP, 5 DEF, 5 MID, 3 FWD, max 3 per team, £{BUDGET:.1f}m budget).')
    lines.append('')
    lines.append(f'**Total cost:** £{total_cost:.1f}m | **Expected points:** {total_ep:.2f} | **Avg FDR:** {avg_fdr:.2f}')
    lines.append('')

    squad_export = squad_df[squad_export_cols].sort_values(
        ['position', 'next_gw_score'],
        key=lambda c: c.map({'GKP': 0, 'DEF': 1, 'MID': 2, 'FWD': 3}) if c.name == 'position' else c,
        ascending=[True, False]
    ).copy()
    squad_export['cost'] = squad_export['cost'].apply(lambda x: f'£{x:.1f}')
    squad_export['fdr_avg'] = squad_export['fdr_avg'].apply(lambda x: f'{x:.2f}')
    squad_export['ep_next'] = squad_export['ep_next'].apply(lambda x: f'{x:.2f}')
    squad_export['points_per_game'] = squad_export['points_per_game'].apply(lambda x: f'{x:.2f}')
    squad_export['next_gw_score'] = squad_export['next_gw_score'].apply(lambda x: f'{x:.2f}')
    lines.append(squad_export.to_markdown(index=False))
    lines.append('')

    # Write file
    content = '\n'.join(lines)
    filename.write_text(content)
    print(f'✓ Saved report to {filename}')
else:
    print('SAVE_OUTPUT is False — skipping export. Set SAVE_OUTPUT = True in config to generate markdown report.')

SAVE_OUTPUT is False — skipping export. Set SAVE_OUTPUT = True in config to generate markdown report.
